# 1. Function Approximation

### 1

Feature engineering defines the representation ϕ(s),
 
while function approximation learns the parameters 𝑤 of a linear model that approximates the value function.

### 2
IPAD

### 3
The policy improvement theorem assumes access to the true action-value function. 
With function approximation we only have an approximation. 
When the policy is improved using 
arg max 𝑄(𝑠,𝑎) the chosen action may not actually be better under the true action-value function. Thus the key inequality 
Qπ(s,π′(s))≥Vπ(s)
used in the proof no longer holds.

### 4
Using a neural network introduces function approximation, while TD(λ) uses bootstrapping, and the data from Mars Rover 1.0 corresponds to off-policy learning. The combination of these three elements is known as the deadly triad, which can lead to instability or divergence during training. As a result, learning the action-value function with a neural network in this setting may fail to converge.


# 2. Planning

### 1
MC v(uni) = 1+1+0+1+1+1+1+0/8 = 6/8 = 0.75
MC v(home)= 0
(G(home)= 0 + lr*0)

### 2
TD v(uni)=6/8 = 0.75
TD v(home)= 0 + lr*v(uni) = 0.75lr

### 3

No, they are not identical.

For uni, both methods give 6/8, but for home:
MC gives 0 but TD gives 0.75*lr
This is because:
MC uses the full observed return from the episode.
TD uses bootstrapping, meaning it updates a state using the estimated value of the next state.
So TD generalizes from the estimate of uni, while MC only uses the one observed return from home.

### 4
To have a full learned model of the game, we need:
the set of states,the set of actions,the transition function and the reward function



### 5 


# Exercise4 Feature Engineering

### 1
Since the state space in MountainCar is continuous (position, velocity), we can convert the problem into a tabular one by discretizing both variables using a fixed step size Δ.

Idea:
Split position and velocity into bins with uniform width.  
Map each continuous state to a discrete index (i, j).  
Then apply standard Q-learning or SARSA on this discrete state.

Example choice:
Δ_{position} = 0.05, Δ_{velocity} = 0.005


This gives a manageable number of discrete states and makes it possible to learn a policy with tabular RL.

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("MountainCar-v0")
low  = env.observation_space.low    # [-1.2, -0.07]
high = env.observation_space.high   # [ 0.6,  0.07]
nA = env.action_space.n

Delta = np.array([0.05, 0.005])  # example: position, velocity step
n_bins = np.floor((high - low) / Delta).astype(int) + 1

def discretize(s):
    idx = np.floor((s - low) / Delta).astype(int)
    return tuple(np.clip(idx, 0, n_bins - 1))

Q = np.zeros((*n_bins, nA))

alpha, gamma = 0.1, 0.99
eps, eps_min, eps_decay = 1.0, 0.02, 0.995
episodes = 3000
max_steps = 1000

for ep in range(episodes):
    s, _ = env.reset()
    ds = discretize(s)

    # Q-learning
    for t in range(max_steps):
        if np.random.rand() < eps:
            a = env.action_space.sample()
        else:
            a = np.argmax(Q[ds])

        s2, r, term, trunc, _ = env.step(a)
        ds2 = discretize(s2)

        td_target = r + gamma * np.max(Q[ds2]) * (not (term or trunc))
        Q[ds + (a,)] += alpha * (td_target - Q[ds + (a,)])

        ds = ds2
        if term or trunc:
            break

    eps = max(eps_min, eps * eps_decay)
